# Lesson 4: Decomposition methods

Neurocampus course "Signals of the whole brain"

Daria Kleeva

dkleeva@gmail.com

March 18, 2026


## Projectors (a bit of abstract and theoretical examples)

Let's take any vector in 2D space. Pick a direction - a unit vector **u**. The projection of **v** onto **u** is a 'footprint' of **v** along that direction. Geometrically, we drop the perpendicular from the tip of **v** onto the line defined by **u**. The projected vector is defined as:

$$\text{proj} = (\text{u}^\text{T} \text{v}) \text{u}$$

The projection matrix is then $$\text{P} = \text{uu}^\text{T}$$

(This formula assumes a unit direction vector). If **u** is not normalized, use this formula:

$$\text{P} = \frac{\text{u} \text{u}^\top}{\text{u}^\top \text{u}} $$

**A glossary of useful terms with simple explanations:**

- Vector — an ordered set of numbers (e.g., sensor values at one time point).
- Vector space — a collection of vectors where addition and scalar multiplication are valid.
- Subspace — a smaller vector space inside a larger one (e.g., artifact direction(s)).
- Span — all linear combinations of given vectors.
- Basis — a minimal set of vectors that spans a space.
- Dimension — number of basis vectors needed to represent the space.
- Linear combination — weighted sum of vectors: `a1 v1 + a2 v2 + ...`
- Projection — mapping a vector onto a subspace.
- Projection matrix — matrix implementing projection.
- Orthogonal — perpendicular in vector-space sense (dot product is zero).
- Orthonormal — orthogonal vectors with unit norm.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

v = np.array([3.0, 2.0])
u = np.array([1.0, 0.5])
u = u / np.linalg.norm(u)  # must be unit vector

P = np.outer(u, u)          # P = u uᵀ
v_proj = P @ v              # projection
v_rej  = v - v_proj         # discarded part, perpendicular to u

fig, ax = plt.subplots(figsize=(5, 5))
ax.axline((0, 0), xy2=u, color='gray', lw=1, ls='--', label='subspace')
ax.quiver(0,0,*v,      angles='xy', scale_units='xy', scale=1, color='steelblue', label='v')
ax.quiver(0,0,*v_proj, angles='xy', scale_units='xy', scale=1, color='tomato',    label='Pv (projection)')
ax.quiver(*v_proj,*v_rej, angles='xy', scale_units='xy', scale=1,
          color='gray', alpha=0.8, label='discarded')
ax.set_xlim(-0.5, 4); ax.set_ylim(-0.5, 3)
ax.set_aspect('equal'); ax.legend(); ax.grid(True)
ax.set_title('Projection onto a 1D subspace')
plt.tight_layout()

<div style="color: blue;">

What happens if the projection is applied twice?
</div>

In [ ]:
#...

If we want to project **away** from the subspace, then we should use **I-P** as a projector:

The projection matrix is then $$\text{P*} = \text{I} - \text{uu}^\text{T}$$

In [ ]:
v = np.array([3.0, 2.0])
u = np.array([1.0, 0.5])
u = u / np.linalg.norm(u)

P      = np.outer(u, u)
v_proj = P @ v
v_rej  = v - v_proj

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# Keep the projection ──
ax = axes[0]
ax.axline((0, 0), xy2=u, color='gray', lw=1, ls='--', label='subspace')
ax.quiver(0, 0, *v,      angles='xy', scale_units='xy', scale=1, color='steelblue', label='v')
ax.quiver(0, 0, *v_proj, angles='xy', scale_units='xy', scale=1, color='tomato',    label='Pv  (kept)')
ax.quiver(*v_proj, *v_rej, angles='xy', scale_units='xy', scale=1, color='gray', alpha=0.4, label='discarded')
ax.set_xlim(-0.5, 4); ax.set_ylim(-0.5, 3)
ax.set_aspect('equal'); ax.legend(); ax.grid(True)
ax.set_title('P = uuᵀ  →  keep the component along u')

# Keep the complement ──
ax = axes[1]
ax.axline((0, 0), xy2=u, color='gray', lw=1, ls='--', label='subspace')
ax.quiver(0, 0, *v,     angles='xy', scale_units='xy', scale=1, color='steelblue', label='v')
ax.quiver(0, 0, *v_rej, angles='xy', scale_units='xy', scale=1, color='seagreen',  label='(I−P)v  (kept)')
ax.quiver(*v_rej, *v_proj, angles='xy', scale_units='xy', scale=1, color='gray', alpha=0.4, label='discarded')
ax.set_xlim(-0.5, 4); ax.set_ylim(-0.5, 3)
ax.set_aspect('equal'); ax.legend(); ax.grid(True)
ax.set_title('I − P  →  keep the component perpendicular to u')

plt.tight_layout()
plt.show()

<div style="color: blue;">

Compare results of applying different projectors. 
</div>

In [ ]:
I = np.eye(2)
P_perp = #...

v_kept    =    # component along u
v_removed =    # component perpendicular to u

print("along u:      ", v_kept)
print("perpendicular:", v_removed)
print("sum restores v:", np.allclose(v_kept + v_removed, v)) 

print((P_perp @ P_perp).round(3))  

So far we projected onto a single direction. What if we want to project onto a plane — a 2D subspace in 3D? Same idea, U is now a matrix whose columns are the basis vectors of that subspace:

$$\text{P} = \text{UU}^\text{T}$$

In [ ]:
U = np.array(###, dtype=float)   # xy-plane in 3D

P_plane = U @ U.T                    # 3x3 projector onto xy-plane
v3 = np.array([2.0, 1.5, 3.0])

print(P_plane @ v3)                
print(np.allclose(P_plane @ P_plane, P_plane))

The z-component vanished because it's perpendicular to the plane. Now let's switch from vectors to time series. Think about EEG. You have N channels and T time points — an N×T matrix. Each column is an N-dimensional vector (the spatial pattern at one moment). A projector acts on that spatial dimension, the same way it acted on our 2D and 3D vectors. It just does it to all T columns at once. 

Here's the key idea for understanding SSP. Suppose we know that blinks "live" in a particular spatial direction — call it **u_artifact**. We found it by looking at blink epochs (or averaging them). Now we want to remove anything in the data that looks like that direction.

In [ ]:
N, T = 8, 1000 
rng = np.random.default_rng(42)
t = np.linspace(0, 2, T)

# Clean signal: sine waves with random spatial mixing
clean_sources = np.vstack([np.sin(2*np.pi*10*t),   # 10 Hz
                            np.sin(2*np.pi*20*t)])  # 20 Hz
A_clean = rng.standard_normal((N, 2))
X_clean = A_clean @ clean_sources

# Artifact: blink = slow deflection with a frontal spatial pattern
u_blink = np.zeros(N)
u_blink[:2] = [0.9, 0.8]                 # strong on frontal channels
u_blink /= np.linalg.norm(u_blink)       # unit vector

blink_shape = np.exp(-((t - 1.0)**2) / 0.01)  # Gaussian bump at t=1s
X_blink = np.outer(u_blink, blink_shape) * 5.0

X_contaminated = X_clean + X_blink

# SSP: project out the artifact subspace ---
P_remove = np.eye(N) - np.outer(u_blink, u_blink)   # I - uuᵀ
X_cleaned = P_remove @ X_contaminated

fig, axes = plt.subplots(3, 1, figsize=(10, 6), sharex=True)
axes[0].plot(t, X_contaminated[0], color='tomato', label='frontal channel — contaminated')
axes[1].plot(t, X_blink[0],        color='orange', label='artifact component alone')
axes[2].plot(t, X_cleaned[0],      color='steelblue', label='after projection')
for ax in axes:
    ax.legend(loc='upper right'); ax.set_yticks([])
axes[2].set_xlabel('time (s)')
plt.suptitle('SSP: remove blink by projecting onto orthogonal complement')
plt.tight_layout()

## Apply SSP projection

<div style="color: blue;">

Let's apply SSP projection we learnt at the previous lesson to MEG data and remove ECG artifacts. MEG data is preloaded below. You will need functions **create_ecg_epochs** and **compute_proj_ecg**.

Use **create_ecg_epochs** to extract epochs time-locked to detected heartbeat events and inspect the average ECG artifact topography. Then use **compute_proj_ecg** to compute SSP projectors that capture the cardiac artifact subspace. Apply the projectors to the raw data and compare the signal before and after artifact removal.

</div>

In [ ]:
import mne
import os
sample_data_folder = mne.datasets.sample.data_path()
sample_data_raw_file = os.path.join(
    sample_data_folder, "MEG", "sample", "sample_audvis_raw.fif"
)
raw = mne.io.read_raw_fif(sample_data_raw_file)

In [ ]:
#...

## Independent component analysis

**Independent Component Analysis (ICA)** is a blind source separation method that assumes observed multichannel signals are linear mixtures of statistically independent latent sources. Formally, we observe $ X = AS $, where $ S $ contains unknown independent components and $ A $ is an unknown mixing matrix; ICA estimates an unmixing matrix $ W \approx A^{-1} $ such that $ \hat{S} = WX $ maximizes statistical independence between components. 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import FastICA
from scipy import signal

Let's create independent sources

In [ ]:
rng = np.random.default_rng(0)
sfreq = 250  # Hz
T = 6.0      # seconds
t = np.arange(int(T * sfreq)) / sfreq
n = t.size

s1 = np.sin(2 * np.pi * 8 * t)                                  # sinusoid (8 Hz)
s2 = signal.sawtooth(2 * np.pi * 2 * t, width=0.3)              # non-sinusoidal wave (2 Hz)
s3 = rng.laplace(loc=0.0, scale=1.0, size=n)        


fig, axs = plt.subplots(3, 1, figsize=(10, 6), sharex=True)

axs[0].plot(t, s1, label='Sinusoid (8 Hz)')
axs[0].set_ylabel('Amplitude')
axs[0].legend(loc='upper right')

axs[1].plot(t, s2, label='Sawtooth (2 Hz)', color='orange')
axs[1].set_ylabel('Amplitude')
axs[1].legend(loc='upper right')

axs[2].plot(t, s3, label='Laplace (0 Hz)', color='green')
axs[2].set_ylabel('Amplitude')
axs[2].set_xlabel('Time (s)')
axs[2].legend(loc='upper right')

plt.tight_layout()

Are the sources Gaussian?

In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(10, 6), sharex=True)

axs[0].hist(s1, bins=20)
axs[0].set_ylabel('Frequency')
axs[0].set_title('Sinusoid (8 Hz)')

axs[1].hist(s2, bins=20)
axs[1].set_ylabel('Frequency')
axs[1].set_title('Sawtooth (2 Hz)')

axs[2].hist(s3, bins=20)
axs[2].set_ylabel('Frequency')
axs[2].set_title('Laplace (0 Hz)')

plt.tight_layout()


from scipy.stats import kurtosis
print(kurtosis(s1))
print(kurtosis(s2))
print(kurtosis(s3))




It is OK to have at most one Gaussian source for ICA. Then, the decomposition is still identifiable. 

Why:
- Gaussian distributions are rotationally symmetric.
- Any orthogonal rotation of Gaussian variables is still Gaussian.
- If we had more than one Gaussian sources, we could rotate them arbitrarily and ICA would have no unique solution.

In [ ]:
S = np.c_[s1, s2, s3]
S = (S - S.mean(axis=0)) / S.std(axis=0)

Now we mix the sources (in real life we do not know the mixing matrix)

In [ ]:
A = np.array([[1.0,  0.5, 0.2],
              [0.3,  1.0, 0.4],
              [0.2, -0.2, 1.0]])  

X = S @ A.T 

#Add some noise to look realistic
X += 0.02 * rng.standard_normal(size=X.shape)

fig, axs = plt.subplots(3, 1, figsize=(10, 6), sharex=True)

axs[0].plot(t, X[:, 0], label='Mixed signal 1')
axs[0].set_ylabel('Amplitude')
axs[0].legend(loc='upper right')

axs[1].plot(t, X[:, 1], label='Mixed signal 2')
axs[1].set_ylabel('Amplitude')
axs[1].legend(loc='upper right')    

axs[2].plot(t, X[:, 2], label='Mixed signal 3')
axs[2].set_ylabel('Amplitude')
axs[2].set_xlabel('Time (s)')
axs[2].legend(loc='upper right')

plt.tight_layout()



Run ICA on mixtures

In [ ]:
ica = FastICA(n_components=3, whiten="unit-variance", random_state=0, max_iter=2000)
S_hat = ica.fit_transform(X)     
A_hat = ica.mixing_   

In [ ]:
A_hat

In [ ]:
A

Let's look how well we recovered the signal

In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(10, 6), sharex=True)

axs[0].plot(t, S_hat[:, 0], label='Reconstructed source 1')
axs[0].set_ylabel('Amplitude')
axs[0].legend(loc='upper right')

axs[1].plot(t, S_hat[:, 1], label='Reconstructed source 2')
axs[1].set_ylabel('Amplitude')
axs[1].legend(loc='upper right')
plt.plot(t, S_hat[:, 2], label='Reconstructed source 3')
plt.legend()
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.tight_layout()

Now let's switch to the real data

In [ ]:
raw = mne.io.read_raw_edf('/Users/dkleeva/Library/CloudStorage/GoogleDrive-dkleeva@gmail.com/My Drive/Teaching/Сигналы целого мозга 2026/Scripts/Data/EEG/probes.edf',
preload=True)
montage = mne.channels.make_standard_montage('standard_1020')
raw.set_montage(montage)
raw.filter(0.1, 40)
import numpy as np

intervals = [
    (0.0, 50.0, "vertical EOG"),
    (52.0, 68.0, "horizontal EOG"),
    (82.0, 89.0, "EMG artefacts"),
    (105.0, 110.0, "Cough artefacts"),
    (216.0, 288.0, "Eyes open"),
    (314.0, 396.0, "Eyes closed")
]

onsets = [start for start, end, lbl in intervals]
durations = [end - start for start, end, lbl in intervals]
descriptions = [lbl for start, end, lbl in intervals]

ann = mne.Annotations(
    onset=onsets,
    duration=durations,
    description=descriptions,
)

raw.set_annotations(ann)


print(raw.annotations)

from collections import defaultdict


segments = defaultdict(list)

for ann in raw.annotations:
    desc = ann["description"]
    tmin = ann["onset"]
    tmax = ann["onset"] + ann["duration"]

    seg = raw.copy().crop(tmin=tmin, tmax=tmax, include_tmax=False)
    segments[desc].append(seg)

raws = {}
for desc, seg_list in segments.items():
    if len(seg_list) == 1:
        raws[desc] = seg_list[0]
    else:
        raws[desc] = mne.concatenate_raws(seg_list)

print(raws.keys())

In [ ]:
raw_for_ica = raws['vertical EOG'].copy()
raw_for_ica.del_proj()

In [ ]:
raw_for_ica.plot()

In [ ]:
from mne.preprocessing import ICA

In [ ]:
ica = ICA(n_components=15, max_iter="auto", random_state=97)
ica.fit(raw_for_ica)

In [ ]:
explained_var_ratio = ica.get_explained_variance_ratio(raw_for_ica)
print(f"Fraction of variance explained by all components: {explained_var_ratio}")


In [ ]:
%matplotlib qt
ica.plot_sources(raw_for_ica)

In [ ]:
%matplotlib inline
ica.plot_components()
plt.show()

In [ ]:
fig = ica.plot_overlay(raw_for_ica.copy().crop(40,45), exclude=[0], picks="eeg")

In [ ]:
ica.plot_properties(raw_for_ica, picks=range(15))

In [ ]:
ica.exclude = []
clean_raw = raw_for_ica.copy()
ica.apply(clean_raw)


In [ ]:
%matplotlib qt
clean_raw.plot()

<div style="color: blue;">

Try:
- Different number of components;
- Different segments;
- Fit ICA on one segment, apply to another;
- Different ICA algorithm

</div>

## Singular Value Decomposition (SVD)

**SVD** factorizes any data matrix $$X \in \mathbb{R}^{m \times n}$$ as

$$ X = U \Sigma V^\top $$

where:
- $U$: left singular vectors (orthonormal, directions in sensor/channel space),
- $\Sigma$: singular values (non-negative, component strengths),
- $V^\top$: right singular vectors (orthonormal, directions in time/sample space).

Intuition: SVD rotates the data into orthogonal components ordered from strongest to weakest.  
Keeping only the first $k$ singular values/vectors gives a low-rank approximation:

$$X_k = U_k \Sigma_k V_k^\top$$

This is useful for denoising, dimensionality reduction, and identifying dominant signal patterns.

In [ ]:

# !pip install -q scikit-image
import numpy as np
import matplotlib.pyplot as plt
from skimage import data, img_as_float

# 1) Load open sample cat image (Chelsea)
img_rgb = img_as_float(data.chelsea())  # shape: (H, W, 3), values in [0, 1]

# Convert to grayscale for simple SVD demo
img = 0.2126 * img_rgb[..., 0] + 0.7152 * img_rgb[..., 1] + 0.0722 * img_rgb[..., 2]
m, n = img.shape

# 2) SVD decomposition
U, s, Vt = np.linalg.svd(img, full_matrices=False)

# 3) SVD spectrum stem plot of first 20 singular values

plt.figure(figsize=(7, 4))
plt.stem(s[:20])
plt.xlabel("k")
plt.ylabel("Singular value")
plt.title("SVD spectrum")
plt.grid(True, alpha=0.3)
plt.show()


# 4) Try different ranks k
k_values = [1,2, 5, 10, 20, 50, 100, 200]

fig, axes = plt.subplots(3, 3, figsize=(14, 9))
axes = axes.ravel()

# Original
axes[0].imshow(img, cmap="gray", vmin=0, vmax=1)
axes[0].set_title(f"Original ({m}x{n})")
axes[0].axis("off")

for i, k in enumerate(k_values, start=1):
    # Rank-k approximation: X_k = U_k * S_k * V_k^T
    Uk = U[:, :k]
    sk = s[:k]
    Vtk = Vt[:k, :]
    img_k = (Uk * sk) @ Vtk
    img_k = np.clip(img_k, 0, 1)

    # Metrics
    mse = np.mean((img - img_k) ** 2)
    rel_err = np.linalg.norm(img - img_k, "fro") / np.linalg.norm(img, "fro")
    compression_ratio = (k * (m + n + 1)) / (m * n)  

    axes[i].imshow(img_k, cmap="gray", vmin=0, vmax=1)
    axes[i].set_title(
        f"k={k}\nMSE={mse:.5f}, rel.err={rel_err:.3f}\nsize~{compression_ratio:.3f}x original"
    )
    axes[i].axis("off")

plt.tight_layout()
plt.show()

# 4) Singular value decay (why small k often works)
energy = np.cumsum(s**2) / np.sum(s**2)
plt.figure(figsize=(7, 4))
plt.plot(energy, lw=2)
plt.xlabel("k")
plt.ylabel("Cumulative energy")
plt.title("How much image energy is captured by first k singular values")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
raw_svd = raws['vertical EOG'].copy().load_data()


X = raw_svd.get_data()
times = raw_svd.times

# Center each channel (important before SVD)
Xc = X - X.mean(axis=1, keepdims=True)


U, s, Vt = np.linalg.svd(Xc, full_matrices=False)
var_ratio = (s**2) / np.sum(s**2)
cum_var = np.cumsum(var_ratio)

print(f"X shape: {Xc.shape}")
print(f"SVD rank: {len(s)}")


n_plot = min(30, len(s))
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

ax[0].stem(np.arange(1, n_plot + 1), s[:n_plot], basefmt=" ")
ax[0].set_title("SVD singular spectrum")
ax[0].set_xlabel("Component index")
ax[0].set_ylabel("Singular value")
ax[0].grid(True, alpha=0.3)

ax[1].plot(np.arange(1, len(cum_var) + 1), cum_var, lw=2)
ax[1].set_title("Cumulative explained variance")
ax[1].set_xlabel("Number of components (k)")
ax[1].set_ylabel("Explained variance ratio")
ax[1].set_ylim(0, 1.02)
ax[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()



In [ ]:

n_topo = min(6, U.shape[1])
fig, axes = plt.subplots(2, 3, figsize=(10, 6))
axes = axes.ravel()

for i in range(6):
    if i < n_topo:
            mne.viz.plot_topomap(
                U[:, i], raw_svd.info, axes=axes[i], show=False, contours=0
            )
            axes[i].set_title(f"Comp {i+1}\nEV={var_ratio[i]*100:.1f}%")

plt.suptitle("SVD spatial patterns (U columns)", y=1.02)
plt.tight_layout()
plt.show()


In [ ]:


n_ts = min(3, len(s))
comp_ts = s[:n_ts, None] * Vt[:n_ts, :]

fig, axs = plt.subplots(n_ts, 1, figsize=(12, 2.2 * n_ts), sharex=True)
if n_ts == 1:
    axs = [axs]

for i in range(n_ts):
    axs[i].plot(times, comp_ts[i], lw=1.0)
    axs[i].set_ylabel(f"C{i+1}")
    axs[i].set_title(f"Temporal activation of component {i+1}")
    axs[i].grid(True, alpha=0.3)

axs[-1].set_xlabel("Time (s)")
plt.tight_layout()
plt.show()


In [ ]:
k_list = [1, 2, 5, 10, 20]
k_list = [k for k in k_list if k <= len(s)]


errs = []
for k in k_list:
    Xk = (U[:, :k] * s[:k]) @ Vt[:k, :]
    rel_err = np.linalg.norm(Xc - Xk, ord="fro") / np.linalg.norm(Xc, ord="fro")
    errs.append(rel_err)

plt.figure(figsize=(6, 4))
plt.plot(k_list, errs, marker='o')
plt.xlabel("k (number of components kept)")
plt.ylabel("Relative reconstruction error")
plt.title("SVD reconstruction error vs k")
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:


ch_idx = 0
tmin = 40.0
tmax = 50.0
sel = (times >= tmin) & (times <= tmax)

plt.figure(figsize=(12, 4))
plt.plot(times[sel], Xc[ch_idx, sel], label="Original (centered)", lw=2, color='k')

for k in k_list:
    Xk = (U[:, :k] * s[:k]) @ Vt[:k, :]
    plt.plot(times[sel], Xk[ch_idx, sel], lw=1.2, label=f"k={k}")

ch_name = raw_svd.info['ch_names'][ch_idx]
plt.title(f"Channel reconstruction with different k: {ch_name}")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.legend(ncol=3, fontsize=9)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()



In [ ]:



plt.figure(figsize=(12, 4))
plt.plot(times[sel], Xc[ch_idx, sel], label="Original (centered)", lw=2, color='k')

for k in k_list:
    Xk = (U[:, k:] * s[k:]) @ Vt[k:, :]
    plt.plot(times[sel], Xk[ch_idx, sel], lw=1.2, label=f"k={k}")

ch_name = raw_svd.info['ch_names'][ch_idx]
plt.title(f"Channel reconstruction without first k components: {ch_name}")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.legend(ncol=3, fontsize=9)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()




## Principal Component Analysis (PCA)

**PCA** is a linear decomposition method that finds orthogonal directions in the data with maximum variance.

PCA is classically defined through the **covariance matrix** of the data:

$$C = \frac{1}{n-1} X^\top X$$

PCA finds the **eigenvectors** of $C$ — the directions along which the data varies most.
Eigenvalues $\lambda_k$ tell you how much variance each direction captures.

PCA finds components such that:
- **PC1** explains the largest possible variance,
- **PC2** explains the next largest variance (orthogonal to PC1),
- and so on.

A common view is:
$$ X \approx T P^\top $$
where:
- $P$: principal axes (spatial patterns / loadings),
- $T$: component scores (time courses / activations).

PCA components are directly related to singular vectors, and explained variance is proportional to $\Sigma^2$.

In [ ]:

from sklearn.decomposition import PCA

raw_pca = raws['vertical EOG'].copy().load_data()

X = raw_pca.get_data()  
times = raw_pca.times
info_picked = mne.pick_info(raw_pca.info, picks)

X_tf = X.T 


pca = PCA(svd_solver='full', random_state=0)
T = pca.fit_transform(X_tf)       # scores, shape (n_times, n_components)
P = pca.components_               # loadings, shape (n_components, n_channels)
evr = pca.explained_variance_ratio_
cum_evr = np.cumsum(evr)

print(f"X shape (channels x time): {X.shape}")
print(f"PCA components: {P.shape[0]}")

n_plot = min(30, len(evr))
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

ax[0].stem(np.arange(1, n_plot + 1), evr[:n_plot], basefmt=" ")
ax[0].set_title("PCA explained variance spectrum")
ax[0].set_xlabel("Component index")
ax[0].set_ylabel("Explained variance ratio")
ax[0].grid(True, alpha=0.3)

ax[1].plot(np.arange(1, len(cum_evr) + 1), cum_evr, lw=2)
ax[1].set_title("PCA cumulative explained variance")
ax[1].set_xlabel("Number of components (k)")
ax[1].set_ylabel("Explained variance ratio")
ax[1].set_ylim(0, 1.02)
ax[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


n_topo = min(6, P.shape[0])
fig, axes = plt.subplots(2, 3, figsize=(10, 6))
axes = axes.ravel()

for i in range(6):
    if i < n_topo:
            topomap_data = P[i]
            mne.viz.plot_topomap(
                topomap_data, info_picked, axes=axes[i], show=False, contours=0
            )
            axes[i].set_title(f"PC{i+1}\nEV={evr[i]*100:.1f}%")


plt.suptitle("PCA spatial loadings (components_)", y=1.02)
plt.tight_layout()
plt.show()

n_ts = min(3, T.shape[1])
fig, axs = plt.subplots(n_ts, 1, figsize=(12, 2.2 * n_ts), sharex=True)
if n_ts == 1:
    axs = [axs]

for i in range(n_ts):
    axs[i].plot(times, T[:, i], lw=1.0)
    axs[i].set_ylabel(f"PC{i+1}")
    axs[i].set_title(f"Temporal score of PC{i+1}")
    axs[i].grid(True, alpha=0.3)

axs[-1].set_xlabel("Time (s)")
plt.tight_layout()
plt.show()


k_list = [1, 2, 5, 10, 20]
k_list = [k for k in k_list if k <= P.shape[0]]

errs = []
X_hat_dict = {}

for k in k_list:
    Tk = T.copy()
    Tk[:, k:] = 0.0
    X_hat_tf = pca.inverse_transform(Tk)  
    X_hat = X_hat_tf.T              
    X_hat_dict[k] = X_hat

    rel_err = np.linalg.norm(X - X_hat, ord='fro') / np.linalg.norm(X, ord='fro')
    errs.append(rel_err)

plt.figure(figsize=(6, 4))
plt.plot(k_list, errs, marker='o')
plt.xlabel("k (number of PCs kept)")
plt.ylabel("Relative reconstruction error")
plt.title("PCA reconstruction error vs k")
plt.grid(True, alpha=0.3)
plt.show()


ch_idx = 0
tmin= 40
tmax =50
sel = (times >= tmin) & (times <= tmax)

plt.figure(figsize=(12, 4))
plt.plot(times[sel], X[ch_idx, sel], label='Original', lw=2, color='k')
for k in k_list:
    plt.plot(times[sel], X_hat_dict[k][ch_idx, sel], lw=1.2, label=f'k={k}')

ch_name = info_picked['ch_names'][ch_idx]
plt.title(f"PCA reconstruction with different k: {ch_name}")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.legend(ncol=3, fontsize=9)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

#reconstruction without first k PCs

plt.figure(figsize=(12, 4))
plt.plot(times[sel], X[ch_idx, sel], label='Original', lw=2, color='k')
for k in k_list:
    Tk = T.copy()
    Tk[:, :k] = 0.0
    X_hat_tf = pca.inverse_transform(Tk)  
    X_hat = X_hat_tf.T              
    X_hat_dict[k] = X_hat

    plt.plot(times[sel], X_hat_dict[k][ch_idx, sel], lw=1.2, label=f'k={k}')

ch_name = info_picked['ch_names'][ch_idx]
plt.title(f"PCA reconstruction without first k PCs: {ch_name}")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.legend(ncol=3, fontsize=9)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()